# Aula 11 — O Transformer e o Treino

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Aviso sobre o tempo: a célula que treina leva alguns minutos no Colab sem
placa de vídeo. É normal. Rode e espere.

## Parte A: Demonstração

### O tokenizador da Aula 9

Nada de novo aqui: é o mesmo código, e o mesmo arquivo de fusões.

In [ ]:
import io
import json
import math
import re
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn

URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
# Alternativa para testar offline:
# URL_DADOS = "../../data/"


def baixar(nome, binario=False):
    if URL_DADOS.startswith("http"):
        dados = urllib.request.urlopen(URL_DADOS + nome).read()
        return dados if binario else dados.decode("utf-8")
    if binario:
        return open(URL_DADOS + nome, "rb").read()
    return open(URL_DADOS + nome, encoding="utf-8").read()


configuracao = json.loads(baixar("machado_bpe.json"))
PADRAO = re.compile(configuracao["padrao"])
fusoes = {(a, b): 256 + i for i, (a, b) in enumerate(configuracao["fusoes"])}

tabela = {i: bytes([i]) for i in range(256)}
for (a, b), novo in fusoes.items():
    tabela[novo] = tabela[a] + tabela[b]

cache = {}


def codificar(texto):
    saida = []
    for pedaco in PADRAO.findall(texto):
        if pedaco not in cache:
            simbolos = list(pedaco.encode("utf-8"))
            while len(simbolos) >= 2:
                candidatos = [p for p in zip(simbolos, simbolos[1:]) if p in fusoes]
                if not candidatos:
                    break
                par = min(candidatos, key=lambda p: fusoes[p])
                novos, i = [], 0
                while i < len(simbolos):
                    if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == par:
                        novos.append(fusoes[par])
                        i += 2
                    else:
                        novos.append(simbolos[i])
                        i += 1
                simbolos = novos
            cache[pedaco] = simbolos
        saida.extend(cache[pedaco])
    return saida


def decodificar(ids):
    return b"".join(tabela[int(i)] for i in ids).decode("utf-8", errors="replace")


corpus = baixar("machado.txt")
tokens = torch.tensor(codificar(corpus), dtype=torch.long)

corte = int(0.9 * len(tokens))
dados_treino, dados_validacao = tokens[:corte], tokens[corte:]

print(f"{len(corpus):,} caracteres viraram {len(tokens):,} tokens")
print(f"treino: {len(dados_treino):,}   validação: {len(dados_validacao):,}")

### Primeiro, o modelo de mentira

Antes de escrever qualquer peça difícil, escreva o modelo inteiro com um
buraco no meio. O bloco recebe e devolve, sem fazer nada. Isso roda, tem
o formato certo, e gera lixo. A partir daqui o trabalho é um só: trocar
o `return x` por algo que valha a pena.

In [ ]:
TAMANHO_VOCABULARIO = 1024
BLOCO = 128        # quantos tokens o modelo enxerga de uma vez
DIMENSAO = 128     # tamanho do vetor de cada token


class BlocoDeMentira(nn.Module):
    def forward(self, x):
        return x                       # de propósito: não faz nada


class ModeloDeMentira(nn.Module):
    def __init__(self):
        super().__init__()
        self.embutir = nn.Embedding(TAMANHO_VOCABULARIO, DIMENSAO)
        self.blocos = nn.ModuleList(BlocoDeMentira() for _ in range(4))
        self.cabeca = nn.Linear(DIMENSAO, TAMANHO_VOCABULARIO, bias=False)

    def forward(self, tokens):
        x = self.embutir(tokens)
        for bloco in self.blocos:
            x = bloco(x)
        return self.cabeca(x)


mentira = ModeloDeMentira()
entrada = torch.tensor([codificar("Não sei se a senhora")])
saida = mentira(entrada)

print(f"entram {tuple(entrada.shape)} tokens")
print(f"saem   {tuple(saida.shape)}: uma pontuação por token do vocabulário")
print(f"pesos: {sum(p.numel() for p in mentira.parameters()):,}")
print()
print("E ele até gera texto:")
contexto = entrada
for _ in range(12):
    proximo = mentira(contexto)[:, -1, :].argmax(dim=-1, keepdim=True)
    contexto = torch.cat([contexto, proximo], dim=1)
print(" ", repr(decodificar(contexto[0].tolist())))
print()
print("Lixo, e lixo com o formato correto. Só falta o miolo.")

### O atalho não é fé: é medida

Antes de montar o bloco de verdade, vale medir por que ele precisa de
atalho. Cinco camadas, um passo para trás, e o gradiente médio que chega
em cada uma.

In [ ]:
class RedeFunda(nn.Module):
    def __init__(self, tamanhos, com_atalho):
        super().__init__()
        self.com_atalho = com_atalho
        self.camadas = nn.ModuleList(
            nn.Sequential(nn.Linear(a, b), nn.GELU())
            for a, b in zip(tamanhos, tamanhos[1:]))

    def forward(self, x):
        for camada in self.camadas:
            saida = camada(x)
            igual = x.shape == saida.shape
            x = x + saida if (self.com_atalho and igual) else saida
        return x


medidas = {}
for com_atalho in [False, True]:
    torch.manual_seed(123)
    rede = RedeFunda([4, 4, 4, 4, 4, 1], com_atalho)
    perda_teste = F.mse_loss(rede(torch.tensor([[1.0, 0.0, -1.0, 1.0]])),
                             torch.zeros(1, 1))
    perda_teste.backward()
    medidas[com_atalho] = [w.grad.abs().mean().item()
                           for nome, w in rede.named_parameters()
                           if "weight" in nome]

print(f"{'camada':<10}{'sem atalho':>14}{'com atalho':>14}")
for i, (sem, com) in enumerate(zip(medidas[False], medidas[True]), 1):
    print(f"{i:<10}{sem:>14.5f}{com:>14.5f}")
print()
print(f"Sem atalho o gradiente encolhe "
      f"{medidas[False][-1] / medidas[False][0]:.0f} vezes até a camada 1.")
print(f"Com atalho, a camada 1 recebe "
      f"{medidas[True][0] / medidas[False][0]:.0f} vezes mais.")

### Agora as peças de verdade

RMSNorm põe cada vetor num tamanho padrão. SwiGLU é a camada densa da
Aula 7, com uma porta em cima. RoPE é o giro da Aula 10. As três são
otimizações: o modelo treinaria sem elas, só um pouco pior.

In [ ]:
CABECAS = 4        # cabeças de atenção por camada
CAMADAS = 4        # blocos empilhados
OCULTA = 256       # largura da camada do meio do MLP


def girar(x, base=10000.0):
    """RoPE: gira o vetor por um ângulo proporcional à posição."""
    lote, cabecas, tokens, dim = x.shape
    metade = dim // 2
    frequencias = base ** (-torch.arange(0, metade).float() / metade)
    angulos = torch.arange(tokens).float()[:, None] * frequencias[None, :]
    cosseno, seno = angulos.cos()[None, None], angulos.sin()[None, None]
    primeira, segunda = x[..., :metade], x[..., metade:]
    return torch.cat([primeira * cosseno - segunda * seno,
                      primeira * seno + segunda * cosseno], dim=-1)


class Atencao(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(DIMENSAO, 3 * DIMENSAO, bias=False)
        self.saida = nn.Linear(DIMENSAO, DIMENSAO, bias=False)

    def forward(self, x):
        lote, tokens, dim = x.shape
        pergunta, chave, valor = self.qkv(x).split(DIMENSAO, dim=2)
        forma = (lote, tokens, CABECAS, dim // CABECAS)
        pergunta, chave, valor = [t.view(forma).transpose(1, 2)
                                  for t in (pergunta, chave, valor)]
        pergunta, chave = girar(pergunta), girar(chave)
        y = F.scaled_dot_product_attention(pergunta, chave, valor, is_causal=True)
        return self.saida(y.transpose(1, 2).reshape(lote, tokens, dim))


class MLP(nn.Module):
    """SwiGLU: dois caminhos, e um deles funciona como porta."""

    def __init__(self):
        super().__init__()
        self.porta = nn.Linear(DIMENSAO, OCULTA, bias=False)
        self.subida = nn.Linear(DIMENSAO, OCULTA, bias=False)
        self.descida = nn.Linear(OCULTA, DIMENSAO, bias=False)

    def forward(self, x):
        return self.descida(F.silu(self.porta(x)) * self.subida(x))


print("peças prontas")

### O bloco, e o atalho

Repare nos dois `x +`. É o atalho (conexão residual): a entrada inteira
atravessa o bloco e é somada à saída.

In [ ]:
class Bloco(nn.Module):
    def __init__(self):
        super().__init__()
        self.norma1 = nn.RMSNorm(DIMENSAO)
        self.norma2 = nn.RMSNorm(DIMENSAO)
        self.atencao = Atencao()
        self.mlp = MLP()

    def forward(self, x):
        x = x + self.atencao(self.norma1(x))
        return x + self.mlp(self.norma2(x))


bloco = Bloco()
entrada_falsa = torch.randn(2, 10, DIMENSAO)
print(f"entra {tuple(entrada_falsa.shape)}, sai {tuple(bloco(entrada_falsa).shape)}")
print(f"pesos de um bloco: {sum(p.numel() for p in bloco.parameters()):,}")

### O modelo inteiro

Embutimento, quatro blocos, uma normalização final e a cabeça de saída.
A cabeça usa **os mesmos pesos** do embutimento, de trás para frente.

In [ ]:
class MiniLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embutir = nn.Embedding(TAMANHO_VOCABULARIO, DIMENSAO)
        self.blocos = nn.ModuleList(Bloco() for _ in range(CAMADAS))
        self.norma = nn.RMSNorm(DIMENSAO)
        self.cabeca = nn.Linear(DIMENSAO, TAMANHO_VOCABULARIO, bias=False)
        self.cabeca.weight = self.embutir.weight   # pesos amarrados
        self.apply(self.iniciar)
        for nome, peso in self.named_parameters():
            if nome.endswith(("saida.weight", "descida.weight")):
                nn.init.normal_(peso, mean=0.0, std=0.02 / (2 * CAMADAS) ** 0.5)

    @staticmethod
    def iniciar(camada):
        """Pesos pequenos no início: sem isso a primeira perda explode."""
        if isinstance(camada, (nn.Linear, nn.Embedding)):
            nn.init.normal_(camada.weight, mean=0.0, std=0.02)

    def forward(self, indices, alvos=None):
        x = self.embutir(indices)
        for bloco in self.blocos:
            x = bloco(x)
        logits = self.cabeca(self.norma(x))
        if alvos is None:
            return logits, None
        perda = F.cross_entropy(logits.view(-1, TAMANHO_VOCABULARIO),
                                alvos.reshape(-1))
        return logits, perda

    @torch.no_grad()
    def gerar(self, indices, quantos, temperatura=0.8, top_k=40):
        for _ in range(quantos):
            logits, _ = self(indices[:, -BLOCO:])
            logits = logits[:, -1, :] / temperatura
            corte = torch.topk(logits, top_k)[0][:, -1:]
            logits = logits.masked_fill(logits < corte, float("-inf"))
            proximo = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            indices = torch.cat([indices, proximo], dim=1)
        return indices


torch.manual_seed(42)
modelo = MiniLLM()
print(f"{sum(p.numel() for p in modelo.parameters()):,} pesos")

### A perda antes de treinar

Sem nenhum treino, o modelo chuta igual entre 1.024 tokens. A perda tem
que dar o logaritmo natural de 1.024:

$$\text{perda inicial} = \ln(1024) = 6{,}93$$

In [ ]:
def sortear(dados, lote=32, gerador=None):
    inicios = torch.randint(len(dados) - BLOCO - 1, (lote,), generator=gerador)
    entrada = torch.stack([dados[i:i + BLOCO] for i in inicios])
    alvo = torch.stack([dados[i + 1:i + BLOCO + 1] for i in inicios])
    return entrada, alvo


entrada, alvo = sortear(dados_treino)
with torch.no_grad():
    _, perda_inicial = modelo(entrada, alvo)

print(f"perda antes de treinar: {perda_inicial:.3f}")
print(f"ln(1024) =              {math.log(1024):.3f}")
print(f"perplexidade: {math.exp(perda_inicial):.0f} candidatos")

### O laço de treino

É o gradiente descendente da Aula 1, com 787.584 pesos em vez de dois.
Vamos rodar poucos passos, só para ver a perda cair.

In [ ]:
otimizador = torch.optim.AdamW(modelo.parameters(), lr=3e-4,
                               betas=(0.9, 0.95), weight_decay=0.1)

inicio = time.time()
historico = []
for passo in range(201):
    entrada, alvo = sortear(dados_treino)
    _, perda = modelo(entrada, alvo)
    otimizador.zero_grad(set_to_none=True)
    perda.backward()
    torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
    otimizador.step()
    historico.append(perda.item())
    if passo % 50 == 0:
        print(f"passo {passo:>4}  perda {perda:.3f}  "
              f"({time.time() - inicio:.0f}s)")

plt.figure(figsize=(9, 4))
plt.plot(historico, color="#B8B8C0", lw=1)
plt.xlabel("Passo")
plt.ylabel("Perda")
plt.title("A perda cai, com barulho por causa do sorteio do lote")
plt.show()

### O modelo treinado de verdade

6.000 passos levariam uns 40 minutos. Este arquivo tem o resultado.

In [ ]:
guardado = torch.load(io.BytesIO(baixar("mini_llm.pt", binario=True)),
                      weights_only=False)

modelo_pronto = MiniLLM()
modelo_pronto.load_state_dict(guardado["pesos"])
modelo_pronto.eval()

print(f"treinado até o passo {guardado['passo']}")
print(f"perda de validação: {guardado['perda_validacao']:.3f}")
print(f"perplexidade: {math.exp(guardado['perda_validacao']):.1f} candidatos")
print()

torch.manual_seed(7)
inicio_texto = torch.tensor([codificar("A casa de")], dtype=torch.long)
saida = modelo_pronto.gerar(inicio_texto, 120)
print(decodificar(saida[0].tolist()))

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: onde moram os pesos

Rode e observe: qual parte do modelo tem mais pesos?

In [ ]:
grupos = {"embutimento": 0, "atenção": 0, "MLP": 0, "normalização": 0}
for nome, peso in modelo.named_parameters():
    if nome.startswith("embutir"):
        grupos["embutimento"] += peso.numel()
    elif "atencao" in nome:
        grupos["atenção"] += peso.numel()
    elif "mlp" in nome:
        grupos["MLP"] += peso.numel()
    else:
        grupos["normalização"] += peso.numel()

total = sum(grupos.values())
for nome, quantos in grupos.items():
    print(f"{nome:<15} {quantos:>8,}  {quantos / total:>5.1%}")
print(f"{'total':<15} {total:>8,}")

In [ ]:
if total == 787_584:
    print("✅ 787.584 pesos. Metade deles está no MLP, não na atenção.")
    print("   A atenção é a peça famosa, mas não é a maior.")
else:
    print(f"❌ Esperava 787.584 e vieram {total}.")

### Exercício 2: o bloco com atalho

Escreva `MeuBloco`, com a mesma estrutura do bloco da aula: normaliza,
atenção, soma de volta; normaliza, MLP, soma de volta.

Dica: são duas linhas dentro do `forward`, e as duas começam com `x +`.

In [ ]:
class MeuBloco(nn.Module):
    def __init__(self):
        super().__init__()
        self.norma1 = nn.RMSNorm(DIMENSAO)
        self.norma2 = nn.RMSNorm(DIMENSAO)
        self.atencao = Atencao()
        self.mlp = MLP()

    def forward(self, x):
        # SEU CODIGO AQUI
        return x

In [ ]:
# SEU CODIGO AQUI

In [ ]:
torch.manual_seed(1)
meu_bloco = MeuBloco()
teste = torch.randn(1, 6, DIMENSAO)
saida_bloco = meu_bloco(teste)
print(f"entra {tuple(teste.shape)}, sai {tuple(saida_bloco.shape)}")

In [ ]:
if saida_bloco.shape == teste.shape and not torch.allclose(saida_bloco, teste):
    print("✅ O bloco devolve o mesmo formato, com o conteúdo mudado.")
    print("   Formato igual é o que permite empilhar quantos você quiser.")
else:
    print("❌ Confira se o forward tem as duas somas com x.")

### Exercício 3: sorteando um lote

Complete `meu_lote`, que devolve entrada e alvo de tamanho
`(lote, BLOCO)`. O alvo é a entrada andada uma casa, como na Aula 9.

In [ ]:
def meu_lote(dados, lote=8):
    inicios = torch.randint(len(dados) - BLOCO - 1, (lote,))
    # SEU CODIGO AQUI
    return None, None

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_entrada, meu_alvo = meu_lote(dados_treino)
print(f"entrada: {tuple(minha_entrada.shape)}")
print(f"alvo:    {tuple(meu_alvo.shape)}")
print()
print("primeira janela, começo:")
print(repr(decodificar(minha_entrada[0][:12].tolist())))
print(repr(decodificar(meu_alvo[0][:12].tolist())))

In [ ]:
if (minha_entrada.shape == (8, BLOCO)
        and torch.equal(minha_entrada[0][1:], meu_alvo[0][:-1])):
    print("✅ Alvo é a entrada deslocada de uma casa.")
    print(f"   Cada lote dá {8 * BLOCO:,} previsões de uma vez.")
else:
    print("❌ Confira os índices: entrada usa i até i+BLOCO, alvo usa i+1 até i+BLOCO+1.")

### Exercício 4: o laço de treino

Complete o corpo do laço. São cinco linhas, sempre nesta ordem:
calcular a perda, zerar os gradientes, retropropagar, cortar o gradiente
e dar o passo.

In [ ]:
torch.manual_seed(42)
meu_modelo = MiniLLM()
meu_otimizador = torch.optim.AdamW(meu_modelo.parameters(), lr=3e-4,
                                   betas=(0.9, 0.95), weight_decay=0.1)
minha_curva = []

In [ ]:
# SEU CODIGO AQUI

In [ ]:
if len(minha_curva) > 100 and minha_curva[-1] < minha_curva[0] - 1.0:
    print(f"✅ A perda caiu de {minha_curva[0]:.2f} para {minha_curva[-1]:.2f}.")
    print("   Em 300 passos o modelo já aprendeu a pontuação e as palavras curtas.")
else:
    print("❌ Se a perda não caiu, confira a ordem das cinco linhas do laço.")

### Exercício 5: a curva de treino

Rode e observe. A linha é cheia de barulho porque cada passo usa um lote
sorteado diferente.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(minha_curva, color="#B8B8C0", lw=1, label="cada passo")
media = [sum(minha_curva[max(0, i - 20):i + 1]) / len(minha_curva[max(0, i - 20):i + 1])
         for i in range(len(minha_curva))]
plt.plot(media, color="#B85042", lw=2.5, label="média dos 20 últimos")
plt.xlabel("Passo")
plt.ylabel("Perda")
plt.legend()
plt.show()

In [ ]:
print("Converse com um colega: por que a linha cinza sobe e desce tanto,")
print("se o modelo está sempre melhorando?")

### Exercício 6: gerando com o seu modelo

Rode e observe o que 300 passos de treino produzem. Não é bonito, e é
esse o ponto: compare com o modelo pronto da Parte A.

In [ ]:
meu_modelo.eval()
torch.manual_seed(7)
comeco = torch.tensor([codificar("A casa de")], dtype=torch.long)
print("SEU MODELO, 300 passos:")
print(decodificar(meu_modelo.gerar(comeco, 80)[0].tolist()))
print()
print("MODELO PRONTO, milhares de passos:")
print(decodificar(modelo_pronto.gerar(comeco, 80)[0].tolist()))
meu_modelo.train()

In [ ]:
print("O seu já aprendeu o tamanho das palavras e onde vai vírgula.")
print("Falta tudo o mais, e o que falta é tempo de processador.")

### Exercício 7: desafio, a perplexidade

Calcule `minha_perplexidade`: a exponencial da perda de validação do seu
modelo. Ela diz entre quantos candidatos o modelo está, na prática,
escolhendo.

$$\text{perplexidade} = e^{\text{perda}}$$

Dica: use `sortear(dados_validacao)` e `math.exp`.

In [ ]:
meu_modelo.eval()

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"perda de validação: {perda_validacao:.3f}")
print(f"perplexidade:       {minha_perplexidade:.0f} candidatos")
print(f"chutar ao acaso:    {TAMANHO_VOCABULARIO} candidatos")
print(f"modelo pronto:      {math.exp(guardado['perda_validacao']):.0f} candidatos")

In [ ]:
if minha_perplexidade < 300:
    print("✅ Bem abaixo dos 1.024 do chute ao acaso.")
    print("   Cada corte pela metade na perplexidade custa muito mais treino que o anterior.")
else:
    print("❌ Confira se você usou math.exp na perda de validação.")

Agora, em texto: a perda de treino do modelo pronto é menor que a de
validação. Explique em duas ou três frases o que essa diferença
significa, e o que aconteceria se ela crescesse muito. Edite esta célula
(duplo clique nela) e escreva sua resposta no lugar deste parágrafo.